# Actividad S3 M9 — 04/09/2025




In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

# tools.py 

# Spark (PySpark)
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F, types as T, Window as W
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import (
    VectorAssembler,
    StringIndexer,
    OneHotEncoder as SparkOneHotEncoder,
    StandardScaler as SparkStandardScaler,
    MinMaxScaler as SparkMinMaxScaler,
    Imputer as SparkImputer,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
)
from pyspark.ml.classification import (
    LogisticRegression as SparkLogisticRegression,
    RandomForestClassifier as SparkRandomForestClassifier,
    GBTClassifier as SparkGBTClassifier,
)
from pyspark.ml.regression import (
    RandomForestRegressor as SparkRandomForestRegressor,
    GBTRegressor as SparkGBTRegressor,
)

def get_spark(app_name: str = "CienciaDeDatos", local_cores: str = "*") -> SparkSession:
    """Crear o recuperar una SparkSession local con nivel de log reducido.

    Parameters
    ----------
    app_name : nombre de la aplicación a mostrar en Spark UI
    local_cores : número de cores locales ("*" usa todos)
    """
    spark = (
        SparkSession.builder
        .master(f"local[{local_cores}]")
        .appName(app_name)
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    return spark




In [2]:
# creacion de conexion

sc= SparkContext("local","testeo")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/04 19:03:31 WARN Utils: Your hostname, Matheus-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.94 instead (on interface en0)
25/09/04 19:03:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/04 19:03:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 64160)
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/Library/Frameworks/Pyt

In [ ]:
# Creacion de la sesión
spark= SparkSession.builder.appName("testeo").getOrCreate()
sc=spark.sparkContext

In [4]:
# insercion de datos
datos=[1,2,3,4,5]
rdd = sc.parallelize(datos)

In [6]:
# ALmacenamiento 
rdd.persist()
rdd.cache()

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297

In [9]:
pares=rdd.filter(lambda x: x%2==0).collect()
print(f"Los números pares son {pares}")

Los números pares son [2, 4]


In [11]:
rdd = sc.parallelize(range(1, 1000000))
pares = rdd.filter(lambda x: x % 2 == 0).count()

25/09/04 21:08:54 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 673610 ms exceeds timeout 120000 ms
25/09/04 21:08:54 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/04 21:08:55 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [10]:
import pyspark.sql.functions as F
from time import time

# Crear un DataFrame grande (ejemplo: 1 millón de filas)
df_big = spark.range(0, 1_000_000).withColumn("valor", (F.rand()*100).cast("int"))

# ⏱️ Sin cache
start = time()
print("Conteo:", df_big.filter(df_big["valor"] > 50).count())
print("Conteo:", df_big.filter(df_big["valor"] > 50).count())  # se vuelve a calcular desde cero
print("Tiempo sin cache:", time() - start)

# ⏱️ Con cache
df_cached = df_big.cache()   # o df_big.persist()
df_cached.count()  # dispara la carga a memoria

start = time()
print("Conteo:", df_cached.filter(df_cached["valor"] > 50).count())
print("Conteo:", df_cached.filter(df_cached["valor"] > 50).count())  # ahora sale rápido
print("Tiempo con cache:", time() - start)

Conteo: 489619
Conteo: 489619
Tiempo sin cache: 1.364243984222412


Conteo: 489619
Conteo: 489619
Tiempo con cache: 0.43131589889526367
